# Data Exploration

In [1]:
!pip install -q python-terrier ir_datasets pandas nltk


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pyterrier as pt
import pandas as pd
import nltk
from nltk.corpus import stopwords
import re

In [5]:
dataset_name = "irds:cord19/trec-covid"
dataset = pt.get_dataset(dataset_name)

print(f"Loaded dataset: {dataset_name}")
irds_dataset = dataset.irds_ref()

Loaded dataset: irds:cord19/trec-covid


## Query Exploration
### Look at Verbosity Levels

In [6]:
topics_data = []
for query in irds_dataset.queries_iter():
    topics_data.append({
        # 'qid': query.query_id,
        'title': query.title,
        'description': query.description,
        'narrative': query.narrative
    })

df_topics = pd.DataFrame(topics_data)
print(f"Total number of queries: {len(df_topics)}")
df_topics.head(3)

Total number of queries: 50


,title,description,narrative
0,coronavirus origin,what is the origin of COVID-19,seeking range of information about the SARS-Co...
1,coronavirus response to weather changes,how does the coronavirus respond to changes in...,seeking range of information about the SARS-Co...
2,coronavirus immunity,will SARS-CoV2 infected people develop immunit...,seeking studies of immunity developed due to i...


### Average Length of the Query at each Verbosity Level Before and After Stopword Removal

In [7]:
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

def count_words(text):
    if pd.isna(text): return 0
    return len(str(text).split())

def count_words_without_stopwords(text):
    if pd.isna(text): return 0
    words = str(text).lower().split()
    filtered_words = [w for w in words if w not in stop_words]
    return len(filtered_words)

In [8]:
verbosity_levels = ['title', 'description', 'narrative']

for level in verbosity_levels:
    df_topics[f'{level}_len'] = df_topics[level].apply(count_words)
    df_topics[f'{level}_len_no_stops'] = df_topics[level].apply(count_words_without_stopwords)


print("--- Average Query Length (INCLUDING Stopwords) ---")
for level in verbosity_levels:
    avg_len = df_topics[f'{level}_len'].mean()
    print(f"{level.capitalize()}: {avg_len:.2f} words")

print("\n--- Average Query Length (EXCLUDING Stopwords) ---")
for level in verbosity_levels:
    avg_len_no_stops = df_topics[f'{level}_len_no_stops'].mean()
    print(f"{level.capitalize()}: {avg_len_no_stops:.2f} words")

--- Average Query Length (INCLUDING Stopwords) ---
Title: 3.24 words
Description: 10.60 words
Narrative: 23.52 words

--- Average Query Length (EXCLUDING Stopwords) ---
Title: 2.74 words
Description: 5.56 words
Narrative: 14.74 words


## Relevance Judgments Exploration

In [9]:
qrels = dataset.get_qrels()

print(f"Total number of relevance judgments: {len(qrels)}")

print("\nRelevance Score Distribution:")
qrels['label'].value_counts()

Total number of relevance judgments: 69318

Relevance Score Distribution:


label
 0    42652
 2    15609
 1    11055
-1        2
Name: count, dtype: int64

## Document Exploration

In [10]:
doc_iter = irds_dataset.docs_iter()

sample_docs = []
for i, doc in enumerate(doc_iter):
    if i >= 5:
        break
    sample_docs.append({
        'docno': doc.doc_id,
        'title': doc.title,
        'abstract': doc.abstract[:300]
    })

df_docs = pd.DataFrame(sample_docs)
df_docs

,docno,title,abstract
0,ug7v899j,Clinical features of culture-proven Mycoplasma...,OBJECTIVE: This retrospective chart review des...
1,02tnwd4m,Nitric oxide: a pro-inflammatory mediator in l...,Inflammatory diseases of the respiratory tract...
2,ejv2xln0,Surfactant protein-D and pulmonary host defense,Surfactant protein-D (SP-D) participates in th...
3,2b73a28n,Role of endothelin-1 in lung disease,Endothelin-1 (ET-1) is a 21 amino acid peptide...
4,9785vg6d,Gene expression in epithelial cells in respons...,Respiratory syncytial virus (RSV) and pneumoni...


# Spell Checking Analysis

In [11]:
!pip install -q pyspellchecker


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import pandas as pd
from spellchecker import SpellChecker
spell = SpellChecker()
def count_spell_errors(text):
    if pd.isna(text): return 0
    text = str(text).lower()
    words = text.split()
    misspelled = spell.unknown(words)
    return len(misspelled)


In [13]:
import re
from spellchecker import SpellChecker

spell = SpellChecker()

verbosity_levels = ['title', 'description', 'narrative']

print("--- Spell Checking Analysis ---\n")

for level in verbosity_levels:
    all_text = " ".join(df_topics[level].dropna().astype(str).tolist()).lower()
    
    words = re.findall(r'\w+', all_text)
    
    flagged_words = spell.unknown(words)
    
    print(f"[{level.capitalize()} Queries]")
    print(f"Total unique flagged words: {len(flagged_words)}")
    
    if len(flagged_words) > 0:
        sample = list(flagged_words)[:20]
        print(f"Sample of flagged words: {', '.join(sample)}\n")

--- Spell Checking Analysis ---

[Title Queries]
Total unique flagged words: 10
Sample of flagged words: covid, cov, hydroxychloroquine, mrna, coronavirus, datasets, biomarkers, cytokine, remdesivir, repurposing

[Description Queries]
Total unique flagged words: 15
Sample of flagged words: covid, cov, hydroxychloroquine, mrna, coronavirus, repurposed, datasets, biomarkers, cytokine, ncov, cov2, triaging, remdesivir, underreporting, angiotensin

[Narrative Queries]
Total unique flagged words: 15
Sample of flagged words: covid, cov, etc, ace2, hydroxychloroquine, cryo, coronavirus, r0, mrna, datasets, biomarkers, cytokine, cov2, remdesivir, angiotensin



In [14]:
import re
from spellchecker import SpellChecker

spell = SpellChecker()

verbosity_levels = ['title', 'description', 'narrative']

print("--- Spell Checking Analysis (With Suggested Corrections) ---\n")

for level in verbosity_levels:
    all_text = " ".join(df_topics[level].dropna().astype(str).tolist()).lower()
    
    words = re.findall(r'\w+', all_text)
    
    flagged_words = spell.unknown(words)
    
    print(f"[{level.capitalize()} Queries]")
    print(f"Total unique flagged words: {len(flagged_words)}")
    
    if len(flagged_words) > 0:
        print("Sample of flagged words -> Suggested correction:")
        sample = list(flagged_words)[:20]
        
        for word in sample:
            suggestion = spell.correction(word)
            print(f"  {word:<15} ->  {suggestion}")
            
    print("-" * 40 + "\n")

--- Spell Checking Analysis (With Suggested Corrections) ---

[Title Queries]
Total unique flagged words: 10
Sample of flagged words -> Suggested correction:
  covid           ->  ovid
  cov             ->  cop
  hydroxychloroquine ->  None
  mrna            ->  mana
  coronavirus     ->  None
  datasets        ->  None
  biomarkers      ->  bookmarkers
  cytokine        ->  cytosine
  remdesivir      ->  None
  repurposing     ->  purposing
----------------------------------------

[Description Queries]
Total unique flagged words: 15
Sample of flagged words -> Suggested correction:
  covid           ->  ovid
  cov             ->  cop
  hydroxychloroquine ->  None
  mrna            ->  mana
  coronavirus     ->  None
  repurposed      ->  purposed
  datasets        ->  None
  biomarkers      ->  bookmarkers
  cytokine        ->  cytosine
  ncov            ->  nov
  cov2            ->  cove
  triaging        ->  trialing
  remdesivir      ->  None
  underreporting  ->  None
  angiotensi

# Relevance Judgements Analysis - complete vs Round 5

In [1]:
import pyterrier as pt

if not pt.started():
    pt.init()

dataset_complete = pt.get_dataset("irds:cord19/trec-covid")
dataset_round5 = pt.get_dataset("irds:cord19/trec-covid/round5")

total_docs = dataset_complete.irds_ref().docs_count()
print(f"Total documents in the corpus: {total_docs}")

qrels_complete = dataset_complete.get_qrels()
qrels_round5 = dataset_round5.get_qrels()

graded_complete_count = qrels_complete['docno'].nunique()
graded_round5_count = qrels_round5['docno'].nunique()

ungraded_complete = total_docs - graded_complete_count
ungraded_round5 = total_docs - graded_round5_count

print("---")
print(f"Complete Dataset:")
print(f"  Graded docs:   {graded_complete_count}")
print(f"  Ungraded docs: {ungraded_complete}")
print("---")
print(f"Round 5 Dataset:")
print(f"  Graded docs:   {graded_round5_count}")
print(f"  Ungraded docs: {ungraded_round5}")

C:\Users\zosia\AppData\Local\Temp\ipykernel_42396\1435902930.py:3: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
C:\Users\zosia\AppData\Local\Temp\ipykernel_42396\1435902930.py:4: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


Total documents in the corpus: 192509
---
Complete Dataset:
  Graded docs:   37924
  Ungraded docs: 154585
---
Round 5 Dataset:
  Graded docs:   17825
  Ungraded docs: 174684


In [5]:
# Count qrels in complete
qrels_complete_counts = qrels_complete['label'].value_counts().sort_index()
print("Complete Dataset Relevance Label Distribution:")

print(qrels_complete_counts)

# count qrels in round 5
qrels_round5_counts = qrels_round5['label'].value_counts().sort_index()
print("\nRound 5 Dataset Relevance Label Distribution:")
print(qrels_round5_counts)


# print total count of qrels in complete and round 5
total_qrels_complete = len(qrels_complete)
total_qrels_round5 = len(qrels_round5)
print("\nTotal Qrels:")
print(f"Complete Dataset: {total_qrels_complete}")
print(f"Round 5 Dataset: {total_qrels_round5}")

Complete Dataset Relevance Label Distribution:
label
-1        2
 0    42652
 1    11055
 2    15609
Name: count, dtype: int64

Round 5 Dataset Relevance Label Distribution:
label
-1        2
 0    12239
 1     4233
 2     6677
Name: count, dtype: int64

Total Qrels:
Complete Dataset: 69318
Round 5 Dataset: 23151
